# Report — Hydropower Unit Commitment (Tokke–Vinje)

**Course:** TDT4173 Modern Machine Learning in Practice — Course Project
**Group number:** `[TODO]`
**Kaggle team name:** `[TODO — must include the group number, e.g. "[34] A Fantastic Team"]`

| Full name | Student ID (from student card) |
|-----------|-------------------------------|
| `[TODO]`  | `[TODO]` |
| `[TODO]`  | `[TODO]` |
| `[TODO]`  | `[TODO]` |

Full write-up of our group work. Clearly titled sections below map directly to the graded items (EDA, ≥2 predictors, feature engineering, model interpretation).


## 0. Executive summary & results

One paragraph: the task, our best model, CV score, public/private LB, and which 2 Kaggle predictions we selected. Fill in the final numbers before submitting.

> **Graders:** this report is **not run** — all outputs below are saved from our actual runs.

In [ ]:
# TODO: code + SAVED OUTPUT for this section (the report is not re-run).


## 1. Problem understanding & domain knowledge  *(EDA item: domain knowledge / how the data was generated)*

### 1.1 What is unit commitment?
A hydropower producer must decide, hour by hour, which of its generating units to run. **Unit commitment** is the binary part of that decision: is unit *g* **ON** (spinning and producing at least its minimum load) or **OFF** in hour *t*? It is a separate decision from *how much* each running unit produces. ON/OFF is a real decision rather than "produce zero" because:

- every unit has a **minimum stable output** `p_min`, which is about **34 % of `p_max`** for almost all units in this system (table below), so a unit cannot run at an arbitrarily low output;
- starting and stopping units costs money and causes wear, so the optimiser prefers continuous ON blocks over rapid switching.

Our task is to predict **P(ON)** for each of the 14 units in each of the 168 hours of a weekly plan. That gives 14 × 168 = 2352 probabilities per case, scored with micro-averaged ROC-AUC.

### 1.2 The Tokke–Vinje watercourse
Tokke–Vinje is a large **cascaded** hydropower system in Telemark, Norway, with **17 reservoirs, 9 plants and 14 units** (≈ 1 GW installed capacity in total). Water used by an upstream plant flows into a downstream reservoir and can be used again further down. The `connections` in `Tokke_Vinje_topology.yaml` give the following main chains:

```
Kjelavatn ─[Vesle Kjela]→ Førsvatn ─[Kjela]→ Hyljelihyl → Venemo → Totak ─┐
Songavatn / Bitdalsvatn ─[Songa]→ Totak ──────────────────────────────────┤
                                         Totak → Våmarvatn ─[Vinje G1–G3]→ Vinjevatn ─[Tokke G1–G4]→ Bandak ─[Hogga]→ (out)
                                         Vatjern ─[Haukeli]→ Vinjevatn
Botnedalsvatn ─[Byrte]→ Byrtevatn ─[Lio]→ r_Vest_Vassdraget → Bandak
```

Every chain ends in **Bandak**, and **Hogga** is the last plant before the water leaves the system. As a result, units are **not independent**: what Vesle Kjela, Kjela and Songa release ends up as storage that Vinje and Tokke can use, and Tokke (4 × 107.5 MW) and Vinje (3 × 100 MW) are the largest plants in the system. Tokke G1–G4 are identical machines, and so are Vinje G1–G3.

### 1.3 How the data was generated
The labels are **not human operating decisions**. They are the optimal schedules computed by **SHOP v17.17.0**, SINTEF's short-term hydropower scheduling optimiser, on the Tokke–Vinje model (v5_2026_6). Each case (`Run No`) is one 168-hour optimisation starting at `starttime`. As the output below shows, **a new case starts every day**, so consecutive cases are rolling weeks that **overlap by 6 of their 7 days**. For each case SHOP is given:

| Input | File | Resolution | Role in the optimisation |
|---|---|---|---|
| Day-ahead price | `Historical_day_ahead_price_2015_2025.csv` | hourly | revenue per MWh, assumed known for the whole week |
| Inflow | `Historical_inflow_1958_2025.csv` | hourly | water entering each reservoir |
| Initial storage | `Historical_volume_2015_2024.csv` | daily (start of day) | volume at the case start |
| Water value | `Synthetic_water_value_2015_2024.csv` | daily | value of water left at the **end** of the week (value at `starttime + 7 days`) |
| Min volume / min flow | `Constraint_min_*.csv` | hourly | environmental and regulatory limits (rarely active) |
| Topology | `Tokke_Vinje_topology.yaml` | static | head curves, efficiencies, `p_min`/`p_max`, connections |

Because the targets come from a **deterministic optimiser**, they are in principle a function of these inputs. The ML task is therefore to learn a fast **surrogate of SHOP**. Two caveats: the water values are *synthetic*, and the volumes of **Hyljelihyl and Vatjern are fixed at 50 %** because no historical records exist, so those two series carry no information.

### 1.4 The economic driver: price vs. water value
The **water value** (EUR/MWh) is the opportunity cost of using water now instead of saving it for later. To a first approximation, **it pays to generate in hour *t* when the price is above the marginal water value**. Water is limited within a week, so the optimiser concentrates production in the **highest-price hours**. A unit's commitment should therefore depend on how an hour's price **ranks within its week**, not only on its absolute level. This simple rule is modified by:

- **Minimum load and start costs:** units tend to run in continuous blocks.
- **Storage and inflow:** a nearly full reservoir with high inflow risks spilling, so the optimiser generates even at low prices. A nearly empty reservoir makes it save water.
- **Constraints:** min-flow and min-volume requirements can force a unit ON (or OFF) regardless of price.
- **Cascade coupling:** downstream plants depend on what upstream plants release.
- **Several identical units per plant:** with identical machines, the optimiser can split a plant's production between them in several equivalent ways.

The code below extracts the system facts used in this section directly from the raw data.

In [1]:
from pathlib import Path
import yaml
import pandas as pd

DATA_DIR = Path("sintef_project/data")   # same config constant as the short notebooks

topo = yaml.safe_load(open(DATA_DIR / "extended" / "Tokke_Vinje_topology.yaml", encoding="utf-8"))
model, conns = topo["model"], topo["connections"]

# --- Units: capacity limits and where each plant discharges its water ---
outlet = {c["from"]: c["to"] for c in conns if c["from_type"] == "plant"}
units = pd.DataFrame([
    {"unit": g, "plant": g.rsplit("_", 1)[0], "p_min [MW]": v["p_min"], "p_max [MW]": v["p_max"]}
    for g, v in model["generator"].items()
])
units["p_min / p_max"] = (units["p_min [MW]"] / units["p_max [MW]"]).round(2)
units["discharges into"] = units["plant"].map(outlet).fillna("(leaves system)")
display(units.set_index("unit"))
print(f"Plants: {units.plant.nunique()}, units: {len(units)}, "
      f"total installed capacity: {units['p_max [MW]'].sum():.1f} MW")

# --- Reservoirs: size (max_vol) ---
res = (pd.Series({r: v["max_vol"] for r, v in model["reservoir"].items()}, name="max_vol [Mm³]")
         .sort_values(ascending=False))
display(res.to_frame().T)

# --- Case structure: rolling weekly optimisations ---
uc = pd.read_csv(DATA_DIR / "kernel" / "Unit_commitment_decisions.csv",
                 usecols=["Run No", "starttime", "total_calculation_time"], parse_dates=["starttime"])
print(f"Training cases: {len(uc)} (Run No {uc['Run No'].min()}–{uc['Run No'].max()}), "
      f"{uc.starttime.min().date()} → {uc.starttime.max().date()}")
print("Days between consecutive case starts:", uc.starttime.diff().dt.days.value_counts().to_dict())
print("Each case = 168 h → consecutive cases overlap by 144 h (6 of 7 days)")
print("SHOP solve time per case [s]:",
      uc.total_calculation_time.describe()[["mean", "min", "max"]].round(1).to_dict())

,plant,p_min [MW],p_max [MW],p_min / p_max,discharges into
unit,,,,,
Hogga_G1,Hogga,3.0,17.0,0.18,(leaves system)
Tokke_G1,Tokke,36.9,107.5,0.34,Bandak
Tokke_G2,Tokke,36.9,107.5,0.34,Bandak
Tokke_G3,Tokke,36.9,107.5,0.34,Bandak
Tokke_G4,Tokke,36.9,107.5,0.34,Bandak
Lio_G1,Lio,14.8,43.0,0.34,r_Vest_Vassdraget
Byrte_G1,Byrte,6.9,20.0,0.34,Byrtevatn
Vinje_G1,Vinje,34.4,100.0,0.34,Vinjevatn
Vinje_G2,Vinje,34.4,100.0,0.34,Vinjevatn


Plants: 9, units: 14, total installed capacity: 1003.4 MW


,Songavatn,Totak,Bordalsvatn,Bandak,Foersvatn,Bitdalsvatn,Kjelavatn,Langesae,Byrtevatn,Botnedalsvatn,Staavatn,Langeidvatn,Vaamarvatn,Venemo,Vinjevatn,Vatjern,Hyljelihyl
max_vol [Mm³],640.0,257.8,184.2,174.2,122.3,110.0,84.5,78.0,76.71,54.17,48.0,32.1,26.2,23.0,9.36,0.3,0.26


Training cases: 2922 (Run No 1–2922), 2015-01-01 → 2022-12-31
Days between consecutive case starts: {1.0: 2921}
Each case = 168 h → consecutive cases overlap by 144 h (6 of 7 days)
SHOP solve time per case [s]: {'mean': 147.0, 'min': 22.4, 'max': 4525.7}


### 1.5 Implications for modelling: hypotheses tested later
The domain analysis gives concrete, testable expectations. Section 2 (EDA) and Section 6 (interpretation) check them:

| # | Hypothesis | Where we check it |
|---|---|---|
| H1 | P(ON) rises with an hour's **within-week price rank** | EDA 2.3, PDP in Section 6 |
| H2 | The **price minus water value** spread is predictive | EDA 2.3, feature importance |
| H3 | High reservoir **fill fraction** / high **inflow** raise ON-rates (spill avoidance) | EDA 2.3 |
| H4 | Strong **hour-of-day** and **seasonal** patterns (spring flood, winter prices) | EDA 2.2 |
| H5 | Units at the same plant are **strongly co-activated**, and identical units may differ in ON-rate | EDA 2.3 |
| H6 | Overlapping daily cases mean **random K-fold leaks information**, so we validate on a **time-based split** | Section 5 |

**Consequences for the pipeline:**
- The model should see *relative* price features (rank and deviation within the case), not only raw price.
- Static topology (capacity, plant, position in the cascade) can be used directly as unit-level features.
- The Hyljelihyl and Vatjern volume columns are constant and should be dropped.

## 2. Exploratory Data Analysis  *(covers ≥4 EDA items → avoids −3)*

Include plots + commentary for each subsection below (this is enough to clear the EDA deduction):

**2.1 Data sanity / is it intuitive** — timestamp coverage & UTC alignment, daily vs hourly granularity, the Hyljelihyl/Vatjern 50%-storage assumption, missing values.

**2.2 Individual features** — price distribution & seasonality; per-unit ON-rate; ON-rate by hour-of-day and season.

**2.3 Pairs / groups of features** — price vs commitment; water value vs commitment; correlations between reservoirs; unit co-activation within a plant.

**2.4 Feature cleanup** — timezone handling, joining daily series onto hourly rows, constant/near-constant constraint columns.

In [ ]:
# TODO: code + SAVED OUTPUT for this section (the report is not re-run).


## 3. Feature Engineering  *(avoids −3)*

Document **extraction** and **selection**:
- Extraction: price rank within the week, rolling price stats, water-value spread (start vs end-of-week), reservoir fill fraction (volume / max_vol from topology), inflow aggregates, upstream/cascade features, unit & plant identity, historical ON-rate priors, calendar features.
- Selection: importance/permutation-based pruning, correlation filter, near-constant removal — report what was dropped and the effect on CV.

In [ ]:
# TODO: code + SAVED OUTPUT for this section (the report is not re-run).


## 4. Modeling — multiple predictor types  *(≥2 types → avoids −3)*

Compare at least two predictor families and present a results table (model × CV ROC-AUC × train time):
- LightGBM (primary)
- XGBoost and/or RandomForest
- Optional: logistic-regression baseline, small MLP/1D-CNN over the weekly sequence.
Explain the long/stacked (Run No, unit, hour) framing and alternatives considered.

**Hyperparameter choices:** show how final values were obtained (CV / grid / Optuna). The short notebooks hard-code these values; this section is the required justification.

In [ ]:
# TODO: code + SAVED OUTPUT for this section (the report is not re-run).


## 5. Validation strategy

Time-based split (early Run No → train, latest pre-test cases → validation) to mirror the 2015–2022 → 2023–2024 shift; why random K-fold leaks across adjacent weeks; the public/private LB gap and why we select final submissions on CV, not public rank.

In [ ]:
# TODO: code + SAVED OUTPUT for this section (the report is not re-run).


## 6. Model Interpretation  *(avoids −3)*

Show at least one (we recommend several):
- LightGBM gain **feature importance**
- **SHAP** summary plot
- **Partial dependence** of P(ON) on price-rank and water-value spread.
Narrative: confirm the model learned the economic rule (higher relative price ⇒ higher P(ON)).

In [ ]:
# TODO: code + SAVED OUTPUT for this section (the report is not re-run).


## 7. Results & leaderboard

Final CV scores, public & private LB scores, the two selected submissions, and how many Virtual Teams we beat. Include the comparison table and any ablation results.

In [ ]:
# TODO: code + SAVED OUTPUT for this section (the report is not re-run).


## 8. Conclusion, limitations & reproducibility

What worked, what didn't (report negative results too — the brief rewards showing all models tried), limitations, and a pointer: *Short notebook 1 reproduces the best score; Short notebook 2 reproduces the second selected submission.*

In [ ]:
# TODO: code + SAVED OUTPUT for this section (the report is not re-run).
